# 🚀 Standalone Cosmetic Ingredient Classifier

This notebook provides a **self-contained prediction environment** for cosmetic ingredient classification with detailed ingredient-level analysis.

## Features:
- ✅ 4-label classification: Halal, Vegan, Allergen-Safe, Eco-Friendly
- 📊 Ingredient-level contribution analysis
- 🧠 62-ingredient knowledge base
- 🤖 BERT + Knowledge Graph hybrid model

## Quick Start:
Run all cells, then use:
```python
predict_ingredients("water, glycerin, niacinamide")
```

## 1️⃣ Imports & Configuration

In [1]:
import os
import re
import glob
import warnings
from typing import Dict, List

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

warnings.filterwarnings('ignore')

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {device}")

# Configuration
MODEL_NAME = 'bert-base-uncased'
KG_FEATURE_DIM = 9
NUM_LABELS = 4
DROPOUT = 0.3

# Label to property mapping
LABEL_TO_PROPERTY = {
    'Halal': 'halal',
    'Vegan': 'vegan',
    'Allergen-Safe': 'allergen',
    'Eco-Friendly': 'eco'
}

print("✅ Configuration loaded")

✅ Device: cpu
✅ Configuration loaded


In [2]:
# Ingredient Knowledge Base with 62 curated ingredients
INGREDIENT_KNOWLEDGE_BASE = {
    # Animal-derived
    'lanolin': {
        'synonyms': ['wool grease', 'wool wax', 'wool fat', 'lanolin alcohol'],
        'chemical_class': 'wax_ester', 'source': 'animal',
        'halal': 0, 'vegan': 0, 'allergen': 0.9, 'eco': 0.2,
        'reasoning': 'Derived from sheep wool; not vegan, strong allergen'
    },
    'collagen': {
        'synonyms': ['hydrolyzed collagen', 'collagen peptides'],
        'chemical_class': 'protein', 'source': 'animal',
        'halal': 0.3, 'vegan': 0, 'allergen': 0.2, 'eco': 0.2,
        'reasoning': 'Animal protein; marine-derived may be halal but not vegan'
    },
    'beeswax': {
        'synonyms': ['cera alba', 'white wax', 'cera flava'],
        'chemical_class': 'wax', 'source': 'animal',
        'halal': 1, 'vegan': 0, 'allergen': 0.1, 'eco': 0.8,
        'reasoning': 'Bee product; halal but not vegan'
    },
    'honey': {
        'synonyms': ['mel', 'bee honey', 'honey extract'],
        'chemical_class': 'natural_sweetener', 'source': 'animal',
        'halal': 1, 'vegan': 0, 'allergen': 0.2, 'eco': 0.8,
        'reasoning': 'Bee product; halal but not vegan'
    },
    'carmine': {
        'synonyms': ['cochineal', 'ci 75470'],
        'chemical_class': 'colorant', 'source': 'animal',
        'halal': 0, 'vegan': 0, 'allergen': 0.3, 'eco': 0.3,
        'reasoning': 'From insects; not halal or vegan'
    },
    'keratin': {
        'synonyms': ['hydrolyzed keratin'],
        'chemical_class': 'protein', 'source': 'animal',
        'halal': 0.2, 'vegan': 0, 'allergen': 0.1, 'eco': 0.2,
        'reasoning': 'Animal protein from hair/feathers'
    },
    'gelatin': {
        'synonyms': ['gelatine'],
        'chemical_class': 'protein', 'source': 'animal',
        'halal': 0.1, 'vegan': 0, 'allergen': 0.1, 'eco': 0.2,
        'reasoning': 'From animal bones/skin'
    },
    'snail mucin': {
        'synonyms': ['snail secretion filtrate', 'snail slime'],
        'chemical_class': 'protein', 'source': 'animal',
        'halal': 0.7, 'vegan': 0, 'allergen': 0.2, 'eco': 0.6,
        'reasoning': 'From snails; not vegan but may be halal'
    },
    'propolis': {
        'synonyms': ['propolis extract'],
        'chemical_class': 'resin', 'source': 'animal',
        'halal': 1, 'vegan': 0, 'allergen': 0.3, 'eco': 0.8,
        'reasoning': 'Bee product; halal but not vegan'
    },

    # Plant-derived
    'shea butter': {
        'synonyms': ['butyrospermum parkii', 'karite butter'],
        'chemical_class': 'plant_oil', 'source': 'plant',
        'halal': 1, 'vegan': 1, 'allergen': 0.05, 'eco': 0.9,
        'reasoning': 'Plant-based, sustainable, very safe'
    },
    'coconut oil': {
        'synonyms': ['cocos nucifera oil', 'copra oil'],
        'chemical_class': 'plant_oil', 'source': 'plant',
        'halal': 1, 'vegan': 1, 'allergen': 0.3, 'eco': 0.8,
        'reasoning': 'Plant oil; potential tree nut allergen'
    },
    'aloe vera': {
        'synonyms': ['aloe barbadensis', 'aloe vera gel', 'aloe barbadensis leaf extract'],
        'chemical_class': 'plant_extract', 'source': 'plant',
        'halal': 1, 'vegan': 1, 'allergen': 0.1, 'eco': 0.95,
        'reasoning': 'Natural plant extract; very safe'
    },
    'jojoba oil': {
        'synonyms': ['simmondsia chinensis', 'jojoba seed oil'],
        'chemical_class': 'plant_oil', 'source': 'plant',
        'halal': 1, 'vegan': 1, 'allergen': 0.05, 'eco': 0.9,
        'reasoning': 'Plant-based liquid wax; very safe'
    },
    'argan oil': {
        'synonyms': ['argania spinosa oil'],
        'chemical_class': 'plant_oil', 'source': 'plant',
        'halal': 1, 'vegan': 1, 'allergen': 0.1, 'eco': 0.85,
        'reasoning': 'Plant oil; safe and sustainable'
    },
    'tea tree oil': {
        'synonyms': ['melaleuca alternifolia oil'],
        'chemical_class': 'essential_oil', 'source': 'plant',
        'halal': 1, 'vegan': 1, 'allergen': 0.4, 'eco': 0.9,
        'reasoning': 'Essential oil; can irritate sensitive skin'
    },
    'rose water': {
        'synonyms': ['rosa damascena water', 'rose hydrosol'],
        'chemical_class': 'plant_extract', 'source': 'plant',
        'halal': 1, 'vegan': 1, 'allergen': 0.2, 'eco': 0.9,
        'reasoning': 'Natural extract; generally safe'
    },
    'green tea extract': {
        'synonyms': ['camellia sinensis extract'],
        'chemical_class': 'plant_extract', 'source': 'plant',
        'halal': 1, 'vegan': 1, 'allergen': 0.1, 'eco': 0.9,
        'reasoning': 'Antioxidant extract; safe'
    },
    'chamomile extract': {
        'synonyms': ['chamomilla recutita extract'],
        'chemical_class': 'plant_extract', 'source': 'plant',
        'halal': 1, 'vegan': 1, 'allergen': 0.2, 'eco': 0.9,
        'reasoning': 'Soothing extract; generally safe'
    },
    'vitamin c': {
        'synonyms': ['ascorbic acid', 'l-ascorbic acid'],
        'chemical_class': 'vitamin', 'source': 'plant',
        'halal': 1, 'vegan': 1, 'allergen': 0.15, 'eco': 0.9,
        'reasoning': 'Plant-derived antioxidant; safe'
    },
    'marula oil': {
        'synonyms': ['sclerocarya birrea seed oil'],
        'chemical_class': 'plant_oil', 'source': 'plant',
        'halal': 1, 'vegan': 1, 'allergen': 0.05, 'eco': 0.9,
        'reasoning': 'Plant-based oil; very safe'
    },
    'glycerin': {
        'synonyms': ['glycerol', 'glycerine'],
        'chemical_class': 'alcohol', 'source': 'plant',
        'halal': 1, 'vegan': 1, 'allergen': 0.02, 'eco': 0.6,
        'reasoning': 'Plant-based (palm/soy); halal and vegan'
    },
    'tocopherol': {
        'synonyms': ['vitamin e', 'tocopheryl acetate'],
        'chemical_class': 'vitamin', 'source': 'plant',
        'halal': 1, 'vegan': 1, 'allergen': 0.05, 'eco': 0.95,
        'reasoning': 'Plant-derived antioxidant; very safe'
    },
    'caffeine': {
        'synonyms': [],
        'chemical_class': 'alkaloid', 'source': 'plant',
        'halal': 1, 'vegan': 1, 'allergen': 0.1, 'eco': 0.85,
        'reasoning': 'Plant extract; safe'
    },

    # Ambiguous
    'stearic acid': {
        'synonyms': ['octadecanoic acid', 'stearate'],
        'chemical_class': 'fatty_acid', 'source': 'ambiguous',
        'halal': 0, 'vegan': 0, 'allergen': 0.05, 'eco': 0.5,
        'reasoning': 'Ambiguous source (animal/plant)'
    },
    'oleic acid': {
        'synonyms': ['decyl oleate', 'sorbitan oleate'],
        'chemical_class': 'fatty_acid', 'source': 'ambiguous',
        'halal': 0, 'vegan': 0.5, 'allergen': 0.05, 'eco': 0.5,
        'reasoning': 'Can be animal/plant-derived'
    },
    'squalene': {
        'synonyms': ['squalane'],
        'chemical_class': 'lipid', 'source': 'ambiguous',
        'halal': 0.5, 'vegan': 0, 'allergen': 0.02, 'eco': 0.5,
        'reasoning': 'Can be shark-derived or plant-derived'
    },
    'lecithin': {
        'synonyms': ['soy lecithin', 'phosphatidylcholine'],
        'chemical_class': 'phospholipid', 'source': 'ambiguous',
        'halal': 0.6, 'vegan': 0.6, 'allergen': 0.3, 'eco': 0.7,
        'reasoning': 'Usually soy-based but can be from eggs'
    },
    'ceramide': {
        'synonyms': ['ceramide np', 'ceramide ap'],
        'chemical_class': 'lipid', 'source': 'ambiguous',
        'halal': 0.5, 'vegan': 0.5, 'allergen': 0.05, 'eco': 0.6,
        'reasoning': 'Can be plant, animal, or synthetic'
    },
    'palmitic acid': {
        'synonyms': ['hexadecanoic acid'],
        'chemical_class': 'fatty_acid', 'source': 'ambiguous',
        'halal': 0.5, 'vegan': 0.5, 'allergen': 0.05, 'eco': 0.4,
        'reasoning': 'From palm oil or animal sources'
    },
    'cetyl alcohol': {
        'synonyms': ['palmityl alcohol'],
        'chemical_class': 'fatty_alcohol', 'source': 'ambiguous',
        'halal': 0.7, 'vegan': 0.7, 'allergen': 0.05, 'eco': 0.7,
        'reasoning': 'Usually plant-based, safe'
    },
    'stearyl alcohol': {
        'synonyms': [],
        'chemical_class': 'fatty_alcohol', 'source': 'ambiguous',
        'halal': 0.7, 'vegan': 0.7, 'allergen': 0.05, 'eco': 0.7,
        'reasoning': 'Usually plant-based, safe'
    },
    'cetearyl alcohol': {
        'synonyms': [],
        'chemical_class': 'fatty_alcohol', 'source': 'ambiguous',
        'halal': 0.7, 'vegan': 0.7, 'allergen': 0.05, 'eco': 0.7,
        'reasoning': 'Mix of cetyl and stearyl; usually safe'
    },
    'fragrance': {
        'synonyms': ['parfum', 'perfume'],
        'chemical_class': 'fragrance', 'source': 'ambiguous',
        'halal': 0.5, 'vegan': 0.5, 'allergen': 0.95, 'eco': 0.3,
        'reasoning': 'Mixed ingredients; major allergen'
    },

    # Synthetic
    'dimethicone': {
        'synonyms': ['polydimethylsiloxane', 'pdms'],
        'chemical_class': 'silicone', 'source': 'synthetic',
        'halal': 1, 'vegan': 1, 'allergen': 0.02, 'eco': 0.2,
        'reasoning': 'Synthetic silicone; not biodegradable'
    },
    'cyclopentasiloxane': {
        'synonyms': ['d5', 'cyclomethicone'],
        'chemical_class': 'silicone', 'source': 'synthetic',
        'halal': 1, 'vegan': 1, 'allergen': 0.02, 'eco': 0.15,
        'reasoning': 'Volatile silicone; environmental concerns'
    },
    'parabens': {
        'synonyms': ['methylparaben', 'propylparaben', 'butylparaben'],
        'chemical_class': 'preservative', 'source': 'synthetic',
        'halal': 1, 'vegan': 1, 'allergen': 0.4, 'eco': 0.2,
        'reasoning': 'Synthetic preservative; controversial'
    },
    'phenoxyethanol': {
        'synonyms': ['phenoxetol'],
        'chemical_class': 'preservative', 'source': 'synthetic',
        'halal': 1, 'vegan': 1, 'allergen': 0.25, 'eco': 0.4,
        'reasoning': 'Synthetic preservative; potential irritant'
    },
    'sodium lauryl sulfate': {
        'synonyms': ['sls', 'sodium dodecyl sulfate'],
        'chemical_class': 'surfactant', 'source': 'synthetic',
        'halal': 1, 'vegan': 1, 'allergen': 0.6, 'eco': 0.3,
        'reasoning': 'Harsh surfactant; can irritate'
    },
    'sodium laureth sulfate': {
        'synonyms': ['sles'],
        'chemical_class': 'surfactant', 'source': 'synthetic',
        'halal': 1, 'vegan': 1, 'allergen': 0.4, 'eco': 0.3,
        'reasoning': 'Milder than SLS but still irritant'
    },
    'peg compounds': {
        'synonyms': ['polyethylene glycol', 'peg-100', 'peg-8'],
        'chemical_class': 'polymer', 'source': 'synthetic',
        'halal': 1, 'vegan': 1, 'allergen': 0.3, 'eco': 0.3,
        'reasoning': 'Synthetic polymers; environmental concerns'
    },
    'petrolatum': {
        'synonyms': ['petroleum jelly'],
        'chemical_class': 'petroleum_derivative', 'source': 'synthetic',
        'halal': 1, 'vegan': 1, 'allergen': 0.05, 'eco': 0.1,
        'reasoning': 'Petroleum-based; not eco-friendly'
    },
    'mineral oil': {
        'synonyms': ['paraffinum liquidum', 'liquid paraffin'],
        'chemical_class': 'petroleum_derivative', 'source': 'synthetic',
        'halal': 1, 'vegan': 1, 'allergen': 0.05, 'eco': 0.1,
        'reasoning': 'Petroleum-based; not biodegradable'
    },
    'paraffin': {
        'synonyms': ['paraffin wax', 'isohexadecane'],
        'chemical_class': 'petroleum_derivative', 'source': 'synthetic',
        'halal': 1, 'vegan': 1, 'allergen': 0.05, 'eco': 0.1,
        'reasoning': 'Petroleum-based wax; not eco-friendly'
    },
    'hyaluronic acid': {
        'synonyms': ['sodium hyaluronate', 'hyaluronan'],
        'chemical_class': 'polysaccharide', 'source': 'synthetic',
        'halal': 1, 'vegan': 1, 'allergen': 0.02, 'eco': 0.95,
        'reasoning': 'Bacterial fermentation; very safe'
    },
    'niacinamide': {
        'synonyms': ['vitamin b3', 'nicotinamide'],
        'chemical_class': 'vitamin', 'source': 'synthetic',
        'halal': 1, 'vegan': 1, 'allergen': 0.05, 'eco': 0.95,
        'reasoning': 'Synthetic vitamin; very safe'
    },
    'retinol': {
        'synonyms': ['vitamin a', 'retinyl palmitate'],
        'chemical_class': 'vitamin', 'source': 'synthetic',
        'halal': 1, 'vegan': 1, 'allergen': 0.35, 'eco': 0.9,
        'reasoning': 'Synthetic; may irritate sensitive skin'
    },
    'salicylic acid': {
        'synonyms': ['bha', 'beta hydroxy acid'],
        'chemical_class': 'hydroxy_acid', 'source': 'synthetic',
        'halal': 1, 'vegan': 1, 'allergen': 0.3, 'eco': 0.85,
        'reasoning': 'Synthetic exfoliant; can irritate'
    },
    'glycolic acid': {
        'synonyms': ['aha', 'alpha hydroxy acid'],
        'chemical_class': 'hydroxy_acid', 'source': 'synthetic',
        'halal': 1, 'vegan': 1, 'allergen': 0.35, 'eco': 0.85,
        'reasoning': 'Synthetic exfoliant; can irritate'
    },
    'lactic acid': {
        'synonyms': [],
        'chemical_class': 'hydroxy_acid', 'source': 'synthetic',
        'halal': 1, 'vegan': 1, 'allergen': 0.3, 'eco': 0.85,
        'reasoning': 'AHA exfoliant; can irritate'
    },
    'peptides': {
        'synonyms': ['palmitoyl pentapeptide', 'matrixyl', 'oligopeptide'],
        'chemical_class': 'peptide', 'source': 'synthetic',
        'halal': 1, 'vegan': 1, 'allergen': 0.05, 'eco': 0.85,
        'reasoning': 'Synthetic peptides; generally safe'
    },
    'alcohol denat': {
        'synonyms': ['denatured alcohol', 'sd alcohol'],
        'chemical_class': 'alcohol', 'source': 'synthetic',
        'halal': 0.1, 'vegan': 1, 'allergen': 0.5, 'eco': 0.6,
        'reasoning': 'Denatured ethanol; not halal, can dry skin'
    },
    'ethanol': {
        'synonyms': ['ethyl alcohol', 'alcohol'],
        'chemical_class': 'alcohol', 'source': 'synthetic',
        'halal': 0.1, 'vegan': 1, 'allergen': 0.45, 'eco': 0.6,
        'reasoning': 'Ethanol; not halal, can dry skin'
    },
    'benzyl alcohol': {
        'synonyms': [],
        'chemical_class': 'aromatic_alcohol', 'source': 'synthetic',
        'halal': 1, 'vegan': 1, 'allergen': 0.3, 'eco': 0.7,
        'reasoning': 'Preservative alcohol; potential irritant'
    },

    # Allergens
    'limonene': {
        'synonyms': ['d-limonene'],
        'chemical_class': 'terpene', 'source': 'plant',
        'halal': 1, 'vegan': 1, 'allergen': 0.7, 'eco': 0.9,
        'reasoning': 'Natural; can cause allergies when oxidized'
    },
    'linalool': {
        'synonyms': ['linalyl alcohol'],
        'chemical_class': 'terpene_alcohol', 'source': 'plant',
        'halal': 1, 'vegan': 1, 'allergen': 0.65, 'eco': 0.9,
        'reasoning': 'Natural fragrance; potential allergen'
    },
    'citral': {
        'synonyms': ['lemonal'],
        'chemical_class': 'terpene', 'source': 'plant',
        'halal': 1, 'vegan': 1, 'allergen': 0.6, 'eco': 0.9,
        'reasoning': 'Citrus fragrance; potential allergen'
    },
    'geraniol': {
        'synonyms': [],
        'chemical_class': 'terpene_alcohol', 'source': 'plant',
        'halal': 1, 'vegan': 1, 'allergen': 0.6, 'eco': 0.9,
        'reasoning': 'Floral fragrance; potential allergen'
    },
    'eugenol': {
        'synonyms': [],
        'chemical_class': 'phenylpropanoid', 'source': 'plant',
        'halal': 1, 'vegan': 1, 'allergen': 0.65, 'eco': 0.9,
        'reasoning': 'Spice fragrance; potential allergen'
    },
    'cinnamal': {
        'synonyms': ['cinnamaldehyde'],
        'chemical_class': 'aldehyde', 'source': 'plant',
        'halal': 1, 'vegan': 1, 'allergen': 0.7, 'eco': 0.85,
        'reasoning': 'Cinnamon fragrance; known allergen'
    },
    'citronellol': {
        'synonyms': [],
        'chemical_class': 'terpene_alcohol', 'source': 'plant',
        'halal': 1, 'vegan': 1, 'allergen': 0.6, 'eco': 0.9,
        'reasoning': 'Natural fragrance; potential allergen'
    },

    # Base ingredients
    'water': {
        'synonyms': ['aqua', 'eau'],
        'chemical_class': 'solvent', 'source': 'natural',
        'halal': 1, 'vegan': 1, 'allergen': 0, 'eco': 1,
        'reasoning': 'Water; completely safe'
    },
    'silica': {
        'synonyms': ['silicon dioxide'],
        'chemical_class': 'mineral', 'source': 'natural',
        'halal': 1, 'vegan': 1, 'allergen': 0.05, 'eco': 0.9,
        'reasoning': 'Natural mineral; safe'
    },
}

print(f"✅ Knowledge Base: {len(INGREDIENT_KNOWLEDGE_BASE)} ingredients")

✅ Knowledge Base: 62 ingredients


## 2️⃣ Ingredient Knowledge Base (62 ingredients)

## 3️⃣ Knowledge Graph Class

In [3]:
# Knowledge Graph for ingredient reasoning
class IngredientKnowledgeGraph:
    def __init__(self, knowledge_base: Dict):
        self.kb = knowledge_base
        self._build_synonym_map()

    def _build_synonym_map(self):
        self.synonym_map = {}
        for ingredient, props in self.kb.items():
            self.synonym_map[ingredient.lower()] = ingredient
            for syn in props.get('synonyms', []):
                self.synonym_map[syn.lower()] = ingredient

    def lookup(self, ingredient_text: str) -> Dict:
        ingredient_lower = ingredient_text.lower().strip()

        if ingredient_lower in self.synonym_map:
            canonical = self.synonym_map[ingredient_lower]
            return {
                'found': True,
                'canonical_name': canonical,
                'properties': self.kb[canonical]
            }

        # Partial match
        for known_ing in self.synonym_map.keys():
            if known_ing in ingredient_lower or ingredient_lower in known_ing:
                canonical = self.synonym_map[known_ing]
                return {
                    'found': True,
                    'canonical_name': canonical,
                    'properties': self.kb[canonical]
                }

        return {
            'found': False,
            'canonical_name': None,
            'properties': {
                'chemical_class': 'unknown', 'source': 'unknown',
                'halal': 0.5, 'vegan': 0.5, 'allergen': 0.5, 'eco': 0.5,
                'reasoning': 'Unknown ingredient'
            }
        }

    def extract_features(self, ingredient_list: List[str]) -> np.ndarray:
        features = np.zeros(9)
        if len(ingredient_list) == 0:
            return features

        found_count = 0
        for ing in ingredient_list:
            result = self.lookup(ing)
            props = result['properties']

            if result['found']:
                found_count += 1

            features[0] += props.get('halal', 0.5)
            features[1] += props.get('vegan', 0.5)
            features[2] += props.get('allergen', 0.5)
            features[3] += props.get('eco', 0.5)

            source = props.get('source', 'unknown')
            if source == 'animal':
                features[4] += 1
            elif source == 'plant':
                features[5] += 1
            elif source == 'synthetic':
                features[6] += 1
            else:
                features[7] += 1

        features[:4] /= len(ingredient_list)
        features[4:8] /= len(ingredient_list)
        features[8] = found_count / len(ingredient_list)

        return features

# Initialize Knowledge Graph
kg = IngredientKnowledgeGraph(INGREDIENT_KNOWLEDGE_BASE)
print(f"✅ Knowledge Graph: {len(kg.synonym_map)} synonym mappings")

✅ Knowledge Graph: 159 synonym mappings


## 4️⃣ Preprocessing Functions

In [4]:
def clean_ingredient_text(text: str) -> str:
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-z0-9,\s\-]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def extract_ingredients(text: str) -> List[str]:
    if not text:
        return []
    ingredients = re.split(r'[,;]', text)
    return [ing.strip() for ing in ingredients if ing.strip()]

print("✅ Preprocessing functions ready")

✅ Preprocessing functions ready


## 5️⃣ Model Architecture

In [5]:
# Hybrid Model: BERT + Knowledge Graph
class HybridCosmeticClassifier(nn.Module):
    def __init__(self, transformer_name='bert-base-uncased',
                 kg_feature_dim=9, num_labels=4, dropout=0.3):
        super().__init__()

        self.transformer = AutoModel.from_pretrained(transformer_name)
        self.transformer_dim = self.transformer.config.hidden_size

        self.kg_encoder = nn.Sequential(
            nn.Linear(kg_feature_dim, 128), nn.LayerNorm(128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 256), nn.LayerNorm(256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 256), nn.LayerNorm(256), nn.ReLU(), nn.Dropout(dropout)
        )

        self.kg_projection = nn.Linear(256, self.transformer_dim)

        self.fusion_attention = nn.MultiheadAttention(
            embed_dim=self.transformer_dim, num_heads=8,
            dropout=dropout, batch_first=True
        )

        self.fusion = nn.Sequential(
            nn.Linear(self.transformer_dim + 256, 768), nn.LayerNorm(768), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(768, 512), nn.LayerNorm(512), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(512, 384), nn.LayerNorm(384), nn.ReLU(), nn.Dropout(dropout)
        )

        self.classifier = nn.Sequential(
            nn.Linear(384, 128), nn.LayerNorm(128), nn.ReLU(), nn.Dropout(dropout/2),
            nn.Linear(128, num_labels)
        )

        self.text_attention = nn.Linear(self.transformer_dim, 1)
        self.kg_attention = nn.Linear(256, 1)

    def forward(self, input_ids, attention_mask, kg_features):
        # Text encoding
        transformer_output = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        text_embedding = transformer_output.last_hidden_state[:, 0, :]

        # KG encoding
        kg_embedding = self.kg_encoder(kg_features)

        # Attention fusion
        text_unsqueezed = text_embedding.unsqueeze(1)
        kg_projected = self.kg_projection(kg_embedding)
        kg_proj_unsqueezed = kg_projected.unsqueeze(1)

        attended_features, _ = self.fusion_attention(
            text_unsqueezed, kg_proj_unsqueezed, kg_proj_unsqueezed
        )
        attended_features = attended_features.squeeze(1)

        # Weighted fusion
        text_attn = torch.sigmoid(self.text_attention(text_embedding))
        kg_attn = torch.sigmoid(self.kg_attention(kg_embedding))
        attn_sum = text_attn + kg_attn + 1e-8

        text_weighted = attended_features * (text_attn / attn_sum)
        kg_weighted = kg_embedding * (kg_attn / attn_sum)

        combined = torch.cat([text_weighted, kg_weighted], dim=1)
        fused = self.fusion(combined)
        logits = self.classifier(fused)

        return logits

print("✅ Model architecture defined")

✅ Model architecture defined


## 6️⃣ Load Tokenizer & Model

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"✅ Tokenizer loaded: {MODEL_NAME}")

# Load model
model_path = '/content/drive/MyDrive/Colab Notebooks/FYP/trained_modals/feb_01_trained_modal/'

def load_model():
    search_paths = [
        model_path,
        './models/',
        './Workaround/'
    ]

    for path in search_paths:
        if os.path.exists(path):
            files = glob.glob(os.path.join(path, '*_epoch_*.pt'))
            if not files:
                files = glob.glob(os.path.join(path, 'best_model.pt'))

            if files:
                latest = max(files, key=lambda f: int(re.search(r'epoch_(\d+)', f).group(1)) if 'epoch' in f else 0)

                model = HybridCosmeticClassifier(
                    transformer_name=MODEL_NAME,
                    kg_feature_dim=KG_FEATURE_DIM,
                    num_labels=NUM_LABELS,
                    dropout=DROPOUT
                ).to(device)

                checkpoint = torch.load(latest, map_location=device)
                model.load_state_dict(checkpoint['model_state_dict'])
                model.eval()

                print(f"✅ Model loaded: {os.path.basename(latest)}")
                return model

    print("❌ Model not found. Please specify path.")
    return None

model = load_model()

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Tokenizer loaded: bert-base-uncased


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded: feb_01_18_44_modal_epoch_7.pt


## 7️⃣ Ingredient Analysis Functions

In [8]:
def analyze_ingredient_contributions(ingredient_list: List[str], label_name: str):
    property_name = LABEL_TO_PROPERTY[label_name]
    positive, negative, neutral = [], [], []
    total_score = 0

    for ing in ingredient_list:
        result = kg.lookup(ing)
        props = result['properties']
        score = props.get(property_name, 0.5)

        if label_name == 'Allergen-Safe':
            score = 1.0 - score  # Invert for allergen

        ing_info = {
            'name': ing,
            'score': score,
            'reasoning': props.get('reasoning', 'Unknown ingredient')
        }

        if score >= 0.7:
            positive.append(ing_info)
        elif score <= 0.3:
            negative.append(ing_info)
        else:
            neutral.append(ing_info)

        total_score += score

    return {
        'positive': positive,
        'negative': negative,
        'neutral': neutral,
        'average_score': total_score / len(ingredient_list) if ingredient_list else 0
    }

print("✅ Analysis functions ready")

✅ Analysis functions ready


## 8️⃣ Prediction Function

In [9]:
def predict_ingredients(ingredient_text: str, verbose=True, show_breakdown=True):
    if model is None:
        print("❌ Model not loaded!")
        return None

    # Preprocess
    clean_text = clean_ingredient_text(ingredient_text)
    ing_list = extract_ingredients(clean_text)

    if verbose:
        print(f"📝 Input: {ingredient_text}")
        print(f"🧪 Detected {len(ing_list)} ingredients\n")

    # Extract features
    kg_feat = kg.extract_features(ing_list)

    # Tokenize
    encoding = tokenizer(clean_text, max_length=256, padding='max_length',
                        truncation=True, return_tensors='pt')

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    kg_tensor = torch.tensor([kg_feat], dtype=torch.float32).to(device)

    # Predict
    model.eval()
    with torch.no_grad():
        logits = model(input_ids, attention_mask, kg_tensor)
        probs = torch.sigmoid(logits).cpu().numpy()[0]

    # Format results
    results = {}
    labels = ['Halal', 'Vegan', 'Allergen-Safe', 'Eco-Friendly']

    if verbose:
        print("="*70)
        print("🎯 PREDICTIONS")
        print("="*70 + "\n")

    for i, label in enumerate(labels):
        prob = float(probs[i])
        prediction = 'Positive' if prob > 0.5 else 'Negative'
        confidence = abs(prob - 0.5) * 2

        analysis = analyze_ingredient_contributions(ing_list, label)

        results[label] = {
            'prediction': prediction,
            'probability': round(prob, 4),
            'confidence': round(confidence, 4),
            'contributors': analysis
        }

        if verbose:
            emoji = "✅" if prediction == 'Positive' else "❌"
            print(f"{emoji} {label:20s}: {prediction:8s} (prob: {prob:.1%}, conf: {confidence:.1%})")

    if verbose:
        print("\n" + "="*70)

        if show_breakdown and ing_list:
            print("\n" + "="*70)
            print("📊 DETAILED INGREDIENT ANALYSIS")
            print("="*70)

            for label in labels:
                analysis = results[label]['contributors']
                print(f"\n┌─ {label.upper()} ANALYSIS")
                print("│")

                if analysis['positive']:
                    print(f"│  ✅ POSITIVE CONTRIBUTORS ({label}-friendly):")
                    for ing in analysis['positive']:
                        print(f"│     • {ing['name']} ({ing['score']*100:.0f}%) - {ing['reasoning']}")

                if analysis['negative']:
                    print("│")
                    print(f"│  ⚠️  NEGATIVE CONTRIBUTORS (Not {label}):")
                    for ing in analysis['negative']:
                        print(f"│     • {ing['name']} ({ing['score']*100:.0f}%) - {ing['reasoning']}")

                if analysis['neutral']:
                    print("│")
                    print("│  ❓ UNKNOWN/AMBIGUOUS:")
                    for ing in analysis['neutral']:
                        print(f"│     • {ing['name']} ({ing['score']*100:.0f}%) - {ing['reasoning']}")

                print("│")
                print(f"└─ ℹ️  Average {label} Score: {analysis['average_score']*100:.0f}%")

            print("\n" + "="*70 + "\n")
        else:
            print("\n")

    return results

print("✅ Prediction function ready!")

✅ Prediction function ready!


## 9️⃣ Example Usage

In [11]:
# # Example 1: Safe ingredients
# print("Example 1: Safe plant-based ingredients\n")
# _ = predict_ingredients("water, glycerin, niacinamide, hyaluronic acid, tocopherol")

print("\n" + "="*80 + "\n")

# Example 2: Contains animal-derived
print("Example 2: Contains animal-derived ingredients\n")
_ = predict_ingredients("water")




Example 2: Contains animal-derived ingredients

📝 Input: water
🧪 Detected 1 ingredients

🎯 PREDICTIONS

✅ Halal               : Positive (prob: 72.7%, conf: 45.3%)
✅ Vegan               : Positive (prob: 72.8%, conf: 45.6%)
✅ Allergen-Safe       : Positive (prob: 65.5%, conf: 30.9%)
✅ Eco-Friendly        : Positive (prob: 68.7%, conf: 37.3%)


📊 DETAILED INGREDIENT ANALYSIS

┌─ HALAL ANALYSIS
│
│  ✅ POSITIVE CONTRIBUTORS (Halal-friendly):
│     • water (100%) - Water; completely safe
│
└─ ℹ️  Average Halal Score: 100%

┌─ VEGAN ANALYSIS
│
│  ✅ POSITIVE CONTRIBUTORS (Vegan-friendly):
│     • water (100%) - Water; completely safe
│
└─ ℹ️  Average Vegan Score: 100%

┌─ ALLERGEN-SAFE ANALYSIS
│
│  ✅ POSITIVE CONTRIBUTORS (Allergen-Safe-friendly):
│     • water (100%) - Water; completely safe
│
└─ ℹ️  Average Allergen-Safe Score: 100%

┌─ ECO-FRIENDLY ANALYSIS
│
│  ✅ POSITIVE CONTRIBUTORS (Eco-Friendly-friendly):
│     • water (100%) - Water; completely safe
│
└─ ℹ️  Average Eco-Friendly 